In [142]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

import joblib

In [143]:
file_path = "../data/Processed/furniture_returns_featured.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (5058, 23)


In [144]:
print("Available columns:\n")

for column in df.columns:
    print("-", column)

Available columns:

- Return ID
- Product
- Value ₹
- Transit Days
- Return Reason
- Condition Hint
- Severity
- Safety
- SLA hrs
- Inspection Outcome
- Resale Before ₹
- Resale After ₹
- Value Loss ₹
- Value Loss %
- Severity Score
- Safety Score
- Condition Risk Score
- Transit Risk Score
- Value Loss Risk Score
- SLA Urgency Score
- Priority Score
- Priority
- Priority Evidence


In [145]:
print("Inspection Outcome Distribution:\n")

print(df["Inspection Outcome"].value_counts())

print("\nPercentage Distribution:")

print(
    (df["Inspection Outcome"]
     .value_counts(normalize=True) * 100)
    .round(2)
)

Inspection Outcome Distribution:

Inspection Outcome
no defect       1955
minor damage    1847
repairable       674
major damage     244
scrap            189
unknown          149
Name: count, dtype: int64

Percentage Distribution:
Inspection Outcome
no defect       38.65
minor damage    36.52
repairable      13.33
major damage     4.82
scrap            3.74
unknown          2.95
Name: proportion, dtype: float64


In [146]:
feature_columns = [
    "Product",
    "Value ₹",
    "Transit Days",
    "Return Reason",
    "Condition Hint",
    "Severity",
    "Safety",
    "SLA hrs",
    "Resale Before ₹",
    "Resale After ₹",
    "Value Loss %",
    "Severity Score",
    "Safety Score",
    "Condition Risk Score",
    "Transit Risk Score",
    "SLA Urgency Score",
    "Value Loss Risk Score"
]

X = df[feature_columns]

print("Number of input features:", len(feature_columns))
print("Input shape:", X.shape)

Number of input features: 17
Input shape: (5058, 17)


In [147]:
y = df["Inspection Outcome"]

print("Target column:")
print(y.name)

print("\nClasses:")
print(y.unique())

Target column:
Inspection Outcome

Classes:
['no defect' 'repairable' 'minor damage' 'scrap' 'unknown' 'major damage']


In [148]:
categorical_features = [
    "Product",
    "Return Reason",
    "Condition Hint",
    "Severity",
    "Safety"
]

numerical_features = [
    "Value ₹",
    "Transit Days",
    "SLA hrs",
    "Resale Before ₹",
    "Resale After ₹",
    "Value Loss %",
    "Severity Score",
    "Safety Score",
    "Condition Risk Score",
    "Transit Risk Score",
    "SLA Urgency Score",
    "Value Loss Risk Score"
]

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Product', 'Return Reason', 'Condition Hint', 'Severity', 'Safety']

Numerical Features:
['Value ₹', 'Transit Days', 'SLA hrs', 'Resale Before ₹', 'Resale After ₹', 'Value Loss %', 'Severity Score', 'Safety Score', 'Condition Risk Score', 'Transit Risk Score', 'SLA Urgency Score', 'Value Loss Risk Score']


In [149]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 4046
Testing samples: 1012


In [150]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [151]:
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight="balanced"
)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", random_forest_model)
    ]
)

print("ML pipeline created successfully!")

ML pipeline created successfully!


In [152]:
model_pipeline.fit(
    X_train,
    y_train
)

print("Model training completed successfully!")

Model training completed successfully!


In [153]:
y_pred = model_pipeline.predict(X_test)

print("Predictions generated successfully!")

Predictions generated successfully!


In [154]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    "Model Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Model Accuracy: 89.43 %


In [155]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

major damage       0.46      0.53      0.50        49
minor damage       0.96      0.96      0.96       369
   no defect       0.97      0.99      0.98       391
  repairable       0.85      0.90      0.87       135
       scrap       0.37      0.37      0.37        38
     unknown       0.00      0.00      0.00        30

    accuracy                           0.89      1012
   macro avg       0.60      0.63      0.61      1012
weighted avg       0.87      0.89      0.88      1012



In [156]:
cm = confusion_matrix(
    y_test,
    y_pred
)

cm

array([[ 26,   0,   0,   5,  18,   0],
       [  1, 354,   1,  11,   1,   1],
       [  0,   0, 389,   0,   0,   2],
       [  6,   4,   0, 122,   3,   0],
       [ 20,   0,   0,   4,  14,   0],
       [  3,  10,  13,   2,   2,   0]])

In [157]:
model = model_pipeline.named_steps["model"]

feature_names = (
    model_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values(
        by="Importance",
        ascending=False
    )
)

feature_importance.head(20)

,Feature,Importance
48,numerical__Value Loss %,0.177367
49,numerical__Severity Score,0.061802
47,numerical__Resale After ₹,0.057485
35,categorical__Severity_high,0.057187
26,categorical__Condition Hint_no visible damage,0.056737
46,numerical__Resale Before ₹,0.053579
54,numerical__Value Loss Risk Score,0.052825
38,categorical__Severity_unknown,0.052707
36,categorical__Severity_low,0.052571
43,numerical__Value ₹,0.051388


In [158]:
df["Predicted Inspection Outcome"] = (
    model_pipeline.predict(X)
)

print(
    df["Predicted Inspection Outcome"]
    .value_counts()
)

Predicted Inspection Outcome
no defect       1959
minor damage    1801
repairable       695
major damage     265
scrap            202
unknown          136
Name: count, dtype: int64


In [159]:
prediction_probabilities = (
    model_pipeline.predict_proba(X)
)

classes = model_pipeline.classes_

probability_df = pd.DataFrame(
    prediction_probabilities,
    columns=[
        f"Probability_{class_name}"
        for class_name in classes
    ]
)

probability_df.head()

,Probability_major damage,Probability_minor damage,Probability_no defect,Probability_repairable,Probability_scrap,Probability_unknown
0,0.000025,0.000417,0.849163,0.000141,0.000000,0.150254
1,0.002143,0.022091,0.000000,0.958983,0.003750,0.013033
2,0.008115,0.636211,0.000556,0.166209,0.005000,0.183909
3,0.049774,0.685435,0.000019,0.063417,0.002475,0.198881
4,0.013534,0.905429,0.002500,0.001833,0.014378,0.062326


In [160]:
df["ML Confidence"] = (
    prediction_probabilities.max(axis=1)
    * 100
).round(2)

df[
    [
        "Return ID",
        "Predicted Inspection Outcome",
        "ML Confidence"
    ]
].head(10)

,Return ID,Predicted Inspection Outcome,ML Confidence
0,R-3047,no defect,84.92
1,R2886,repairable,95.90
2,R2747,minor damage,63.62
3,R4670,minor damage,68.54
4,R4898,minor damage,90.54
5,R2596,scrap,42.41
6,R3729,scrap,71.45
7,R3755,minor damage,92.45
8,R3354,repairable,82.04
9,r79,no defect,69.83


In [161]:
ml_results = df[
    [
        "Return ID",
        "Product",
        "Inspection Outcome",
        "Predicted Inspection Outcome",
        "ML Confidence"
    ]
]

ml_results.head(20)

,Return ID,Product,Inspection Outcome,Predicted Inspection Outcome,ML Confidence
0,R-3047,bed,no defect,no defect,84.92
1,R2886,bed,repairable,repairable,95.90
2,R2747,unknown,minor damage,minor damage,63.62
3,R4670,dining table,minor damage,minor damage,68.54
4,R4898,dining table,minor damage,minor damage,90.54
5,R2596,bed,repairable,scrap,42.41
6,R3729,sofa,scrap,scrap,71.45
7,R3755,wardrobe,minor damage,minor damage,92.45
8,R3354,coffee table,repairable,repairable,82.04
9,r79,chair,no defect,no defect,69.83


In [162]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

model_path = "../models/random_forest_inspection_model.pkl"

joblib.dump(
    model_pipeline,
    model_path
)

print("Model saved successfully!")
print(model_path)

Model saved successfully!
../models/random_forest_inspection_model.pkl


In [163]:
output_path = "../data/Processed/furniture_returns_ml_predictions.xlsx"

df.to_excel(
    output_path,
    index=False
)

print("ML prediction results saved successfully!")
print(output_path)

ML prediction results saved successfully!
../data/Processed/furniture_returns_ml_predictions.xlsx


In [164]:
print("=" * 60)
print("MACHINE LEARNING MODEL SUMMARY")
print("=" * 60)

print("\nAlgorithm:")
print("Random Forest Classifier")

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))

print("\nAccuracy:")
print(round(accuracy * 100, 2), "%")

print("\nTarget Classes:")
print(list(model_pipeline.classes_))

print("\nTop 10 Important Features:")
print(feature_importance.head(10).to_string(index=False))

MACHINE LEARNING MODEL SUMMARY

Algorithm:
Random Forest Classifier

Training Samples: 4046
Testing Samples: 1012

Accuracy:
89.43 %

Target Classes:
['major damage', 'minor damage', 'no defect', 'repairable', 'scrap', 'unknown']

Top 10 Important Features:
                                      Feature  Importance
                      numerical__Value Loss %    0.177367
                    numerical__Severity Score    0.061802
                    numerical__Resale After ₹    0.057485
                   categorical__Severity_high    0.057187
categorical__Condition Hint_no visible damage    0.056737
                   numerical__Resale Before ₹    0.053579
             numerical__Value Loss Risk Score    0.052825
                categorical__Severity_unknown    0.052707
                    categorical__Severity_low    0.052571
                           numerical__Value ₹    0.051388
